# 05 - Modelling: MLP (v2)

Trains a small Multi-Layer Perceptron on the same 20-feature v2 schema, as a deep-learning comparison point against Random Forest, XGBoost, and LightGBM from `04_modelling_classical.ipynb`. Reproduces `04`'s exact split inline (same fixed `RANDOM_STATE`, same source `dataset_pe_v2_full.csv`), the same reproducibility approach `legacy_v1` uses (no split files saved to disk). Test set is not touched, model selection happens in `06_evaluation.ipynb`.

**Not executed in this sandbox** (no TensorFlow installed here). Run locally: `Kernel -> Restart & Run All`.

In [8]:
# Standard library
import sys

# Third-party
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Model, layers

pd.set_option("display.max_columns", None)
sys.path.append("../src")
from constants import ORDER_OF_FEATURES, LABEL_COLUMN, RANDOM_STATE

tf.random.set_seed(RANDOM_STATE)

**Compute-constrained downsample of the training pool (2026-07, added after real local testing).** `GridSearchCV`'s exhaustive search over the full ~600,000-row training pool proved too slow on typical laptop hardware (each of the three grid searches trains many separate models across cross-validation folds). Downsamples `train_pool` to 75,000 rows per class (150,000 total, still roughly 7-8x larger than v1's ~19,600-row dataset) before the 85/15 train/validation split, using a fixed `RANDOM_STATE` so the reduction is reproducible. **The held-out `test` set (EMBER's own official 200,000-row split) is NOT reduced**: `06_evaluation.ipynb`'s final reported numbers still come from the complete, standard benchmark split, only the expensive tuning step works on a smaller pool. This is a disclosed, documented tradeoff for real hardware constraints, not a silent change.

In [9]:
df = pd.read_csv("../data/dataset_pe_v2_full.csv")
train_pool = df[df["OriginalSplit"] == "train"].reset_index(drop=True)

TRAIN_POOL_SAMPLE_PER_CLASS = 75000
train_pool = pd.concat([
    g.sample(n=min(TRAIN_POOL_SAMPLE_PER_CLASS, len(g)), random_state=RANDOM_STATE)
    for _, g in train_pool.groupby(LABEL_COLUMN)
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print("downsampled train_pool:", train_pool.shape)

train_df = pd.concat([
    g.sample(frac=0.85, random_state=RANDOM_STATE)
    for _, g in train_pool.groupby(LABEL_COLUMN)
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
val_df = train_pool.drop(
    pd.concat([g.sample(frac=0.85, random_state=RANDOM_STATE) for _, g in train_pool.groupby(LABEL_COLUMN)]).index
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_train, y_train = train_df[ORDER_OF_FEATURES], train_df[LABEL_COLUMN]
X_val, y_val = val_df[ORDER_OF_FEATURES], val_df[LABEL_COLUMN]
print("train:", X_train.shape, "val:", X_val.shape)

downsampled train_pool: (150000, 23)
train: (127500, 20) val: (22500, 20)


## Scale the features

Fit once on `X_train` only, the same leakage-safe rule `04`'s pipelines enforce automatically; here it's manual since a neural network isn't wrapped in a scikit-learn `Pipeline`.

In [10]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)

## Build the MLP

Input (20 features) -> 128 -> 64 -> 1 output (probability of malicious). Same shape philosophy as v1, widened input layer only because there are twice as many input features now.

In [11]:
inputs = layers.Input(shape=(X_train_scaled.shape[1],))
x = layers.Dense(128, activation="relu")(inputs)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 20)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 128)                 │           2,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 11,009 (43.00 KB)

 Trainable params: 11,009 (43.00 KB)

 Non-trainable params: 0 (0.00 B)

## Train the model

Classes are already exactly balanced (50/50), so no `class_weight` correction is needed here, unlike v1. `EarlyStopping` stops once validation performance stalls for 3 epochs and restores the best weights seen.

In [12]:
early_stop = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train.values,
    validation_data=(X_val_scaled, y_val.values),
    epochs=30, batch_size=256,
    callbacks=[early_stop],
    verbose=2,
)

Epoch 1/30
499/499 - 3s - 6ms/step - accuracy: 0.7586 - loss: 0.5180 - val_accuracy: 0.8080 - val_loss: 0.4299
Epoch 2/30
499/499 - 2s - 5ms/step - accuracy: 0.8055 - loss: 0.4390 - val_accuracy: 0.8286 - val_loss: 0.4008
Epoch 3/30
499/499 - 3s - 5ms/step - accuracy: 0.8205 - loss: 0.4062 - val_accuracy: 0.8375 - val_loss: 0.3651
Epoch 4/30
499/499 - 3s - 5ms/step - accuracy: 0.8277 - loss: 0.3862 - val_accuracy: 0.8443 - val_loss: 0.3500
Epoch 5/30
499/499 - 3s - 5ms/step - accuracy: 0.8334 - loss: 0.3709 - val_accuracy: 0.8482 - val_loss: 0.3394
Epoch 6/30
499/499 - 3s - 5ms/step - accuracy: 0.8366 - loss: 0.3619 - val_accuracy: 0.8525 - val_loss: 0.3357
Epoch 7/30
499/499 - 3s - 5ms/step - accuracy: 0.8415 - loss: 0.3541 - val_accuracy: 0.8557 - val_loss: 0.3251
Epoch 8/30
499/499 - 3s - 5ms/step - accuracy: 0.8439 - loss: 0.3483 - val_accuracy: 0.8594 - val_loss: 0.3193
Epoch 9/30
499/499 - 3s - 5ms/step - accuracy: 0.8464 - loss: 0.3417 - val_accuracy: 0.8620 - val_loss: 0.3139
E

## Evaluate on the validation set

In [13]:
val_preds = (model.predict(X_val_scaled) >= 0.5).astype(int).ravel()
print(classification_report(y_val, val_preds, target_names=["benign", "malicious"]))

704/704 ━━━━━━━━━━━━━━━━━━━━ 1s 802us/step
              precision    recall  f1-score   support

      benign       0.89      0.88      0.89     11250
   malicious       0.88      0.89      0.89     11250

    accuracy                           0.89     22500
   macro avg       0.89      0.89      0.89     22500
weighted avg       0.89      0.89      0.89     22500



## Save the MLP candidate and its scaler

Keras's own native format, matching v1's convention. The scaler is saved separately (joblib) since it isn't part of the Keras model itself but is required to preprocess any new input the same way at inference time.

In [14]:
import joblib
model.save("../models/mlp_v2.keras")
joblib.dump(scaler, "../models/mlp_v2_scaler.joblib")
print("saved mlp_v2.keras and mlp_v2_scaler.joblib to ../models/")

saved mlp_v2.keras and mlp_v2_scaler.joblib to ../models/


## Summary

TODO (fill in after a real local run): validation precision/recall/F1 for the MLP, and how it compares to Random Forest/XGBoost/LightGBM from `04`. Next: `06_evaluation.ipynb` compares all four candidates on the held-out test set and picks the final deployed model.